In [1]:
import os, re, random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from bert_score import score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


/Users/shannon/miniconda3/envs/minienv/lib/python3.11/site-packages/sklearn/utils/__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.3)
  from scipy.sparse import issparse


In [2]:
# CONFIG
BASE_DIR       = Path.home() / 'Desktop' / 'Few-Shot Experiments (DeepSeek-Chat)'
LEGAL_BERT     = '/Users/shannon/legal-bert'
NUM_LAYERS     = 9
TERM_WEIGHT    = 0.5
BERT_WEIGHT    = 0.5
NUM_ITERATIONS = 50
RANDOM_SEED    = 42
OUTPUT_FILE    = 'FewShot_KeyPrinciples_Issue_Results.xlsx'

CASE_GROUPS_3 = {
    'GROUP 1: Corporate, Commercial & Insolvency':    [2, 22, 24, 8, 15, 17, 10, 16, 14, 19, 18, 20],
    'GROUP 2: Public Law & Regulation':               [5, 12, 4, 11, 7, 9],
    'GROUP 3: Employment, Personal & Civil Disputes': [6, 13, 21, 23, 25, 3, 1],
}

SHOT_SETTINGS = ['Zero-Shot', '5-Shot', '10-Shot', '15-Shot', '20-Shot']

TASKS = {
    'KeyPrinciple': {
        'folder_pattern': '{shot}_individual_keyprinciple_extraction_results',
        'file_pattern':   'Case_{id}_KeyPrinciple_Extraction.xlsx',
        'pred_col':       'KeyPrinciple_DEEPSEEK',
        'ref_col':        'KeyPrinciple_Annotated',
    },
    'Issue': {
        'folder_pattern': '{shot}_individual_issue_extraction_results',
        'file_pattern':   'Case_{id}_Extraction.xlsx',
        'pred_col':       'Issue_DEEPSEEK',
        'ref_col':        'Issue_Annotated',
    },
}


In [3]:
# HELPERS (identical to LLM Selection notebook)
def extract_legal_terms(text, min_length=3):
    if not text or pd.isna(text):
        return set()
    text = str(text).lower()
    stopwords = {'the','a','an','and','or','but','in','on','at','to','for',
                 'of','with','by','from','as','is','was','be','been','are',
                 'has','have','had','that','which','who','whom','this','these',
                 'those','it','its'}
    words = re.findall(r'\b[a-z]+\b', text)
    terms = set()
    for word in words:
        if len(word) >= min_length and word not in stopwords:
            terms.add(word)
    words_list = [w for w in words if w not in stopwords]
    for i in range(len(words_list) - 1):
        terms.add(f'{words_list[i]} {words_list[i+1]}')
    for i in range(len(words_list) - 2):
        trigram = f'{words_list[i]} {words_list[i+1]} {words_list[i+2]}'
        if any(w in trigram for w in ['natural justice','breach','duty','reasonable','tribunal']):
            terms.add(trigram)
    return terms

def calculate_term_overlap(text1, text2):
    if not text1 or not text2 or pd.isna(text1) or pd.isna(text2):
        return 0.0
    terms1 = extract_legal_terms(text1)
    terms2 = extract_legal_terms(text2)
    if not terms1 or not terms2:
        return 0.0
    intersection = terms1.intersection(terms2)
    union = terms1.union(terms2)
    jaccard = len(intersection) / len(union) if union else 0.0
    key_phrases = [
        'natural justice','breach of duty','reasonable care',
        'contractual obligation','statutory interpretation',
        'constitutional challenge','judicial review','abuse of process',
        'burden of proof','standard of proof','procedural fairness',
        'substantive fairness','legitimate expectation'
    ]
    bonus = sum(0.15 for p in key_phrases if p in str(text1).lower() and p in str(text2).lower())
    return min(jaccard + bonus, 1.0)

def sample_one_iteration(folder, file_pattern, ref_col, pred_col, case_groups, iteration, seed):
    random.seed(seed)
    np.random.seed(seed)
    sampled = []
    for group_name, case_list in case_groups.items():
        shuffled = case_list.copy()
        random.shuffle(shuffled)
        for case_id in shuffled:
            case_file = folder / file_pattern.format(id=case_id)
            if not case_file.exists():
                continue
            try:
                df = pd.read_excel(case_file)
                df_valid = df[
                    df[ref_col].notna() & df[pred_col].notna() &
                    (df[ref_col].astype(str).str.strip() != '') &
                    (df[pred_col].astype(str).str.strip() != '')
                ]
                if len(df_valid) == 0:
                    continue
                row = df_valid.sample(n=1, random_state=np.random.randint(0, 100000)).copy()
                row['Source_Case']  = case_id
                row['Source_Group'] = group_name
                sampled.append(row)
                break
            except Exception:
                continue
    if len(sampled) == len(case_groups):
        return pd.concat(sampled, ignore_index=True)
    return None

def calculate_accuracy(df_sample, ref_col, pred_col):
    annotated = [str(x) if pd.notna(x) else '' for x in df_sample[ref_col].tolist()]
    predicted = [str(x) if pd.notna(x) else '' for x in df_sample[pred_col].tolist()]
    n = len(annotated)
    term_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            term_mat[i, j] = calculate_term_overlap(predicted[i], annotated[j])
    f1_mat = np.zeros((n, n))
    for col_idx in range(n):
        refs = [annotated[col_idx]] * n
        _, _, F1 = score(predicted, refs,
                         model_type=LEGAL_BERT, num_layers=NUM_LAYERS,
                         verbose=False,
                         device='cuda' if torch.cuda.is_available() else 'cpu')
        f1_mat[:, col_idx] = F1.numpy()
    hybrid = (TERM_WEIGHT * term_mat) + (BERT_WEIGHT * f1_mat)
    correct = sum(np.argmax(hybrid[:, j]) == j for j in range(n))
    return correct, n


In [4]:
# MAIN LOOP
SEP = '=' * 70
all_results  = []
summary_rows = []

for task, cfg in TASKS.items():
    for shot in SHOT_SETTINGS:
        folder = BASE_DIR / cfg['folder_pattern'].format(shot=shot)
        print(SEP)
        print(f'TASK: {task} | SHOT: {shot}')
        print(SEP)
        if not folder.exists():
            print(f'  Folder not found: {folder}')
            continue

        total_correct, total_n, perfect = 0, 0, 0

        for iteration in range(1, NUM_ITERATIONS + 1):
            seed = RANDOM_SEED + iteration
            df_sample = sample_one_iteration(
                folder, cfg['file_pattern'], cfg['ref_col'], cfg['pred_col'],
                CASE_GROUPS_3, iteration, seed
            )
            if df_sample is None:
                print(f'  Iteration {iteration}: sampling failed')
                continue
            correct, n = calculate_accuracy(df_sample, cfg['ref_col'], cfg['pred_col'])
            total_correct += correct
            total_n       += n
            if correct == n:
                perfect += 1
            if iteration % 10 == 0:
                running = total_correct / total_n * 100
                print(f'  Iteration {iteration}/{NUM_ITERATIONS} — running acc: {running:.1f}%')
            all_results.append({
                'Task': task, 'Shot': shot, 'Iteration': iteration,
                'Correct': correct, 'Total': n,
                'Accuracy_%': round(100*correct/n, 1) if n else 0,
                'Perfect': 'Yes' if correct == n else 'No'
            })

        acc       = round(100 * total_correct / total_n, 1) if total_n else 0
        waterfall = round(100 * perfect / NUM_ITERATIONS, 1)
        print(f'  >> {shot}: {total_correct}/{total_n} = {acc}% | Waterfall: {waterfall}%')
        summary_rows.append({
            'Task': task, 'Shot': shot,
            'Total_Correct': total_correct, 'Total_Samples': total_n,
            'Accuracy_%': acc, 'Perfect_Iterations': perfect,
            'Waterfall_Rate_%': waterfall,
        })


TASK: KeyPrinciple | SHOT: Zero-Shot


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
# SUMMARY
SEP = '=' * 70
results_df = pd.DataFrame(summary_rows)

print(SEP)
print('FINAL SUMMARY')
print(SEP)
for task in TASKS:
    print(f'\n-- {task} --')
    sub = results_df[results_df['Task']==task][
        ['Shot','Total_Correct','Total_Samples','Accuracy_%','Perfect_Iterations','Waterfall_Rate_%']
    ]
    print(sub.to_string(index=False))

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as w:
    results_df.to_excel(w, sheet_name='Summary', index=False)
    pd.DataFrame(all_results).to_excel(w, sheet_name='Full_Results', index=False)

print(f'\nSaved to {OUTPUT_FILE}')
print(f'Completed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


FINAL SUMMARY

-- KeyPrinciple --
     Shot  Total_Correct  Total_Samples  Accuracy_%  Perfect_Iterations  Waterfall_Rate_%
Zero-Shot            130            150        86.7                  32              64.0
   5-Shot            127            150        84.7                  30              60.0
  10-Shot            128            150        85.3                  30              60.0
  15-Shot            130            150        86.7                  33              66.0
  20-Shot            130            150        86.7                  33              66.0

-- Issue --
     Shot  Total_Correct  Total_Samples  Accuracy_%  Perfect_Iterations  Waterfall_Rate_%
Zero-Shot            112            150        74.7                  18              36.0
   5-Shot            115            150        76.7                  22              44.0
  10-Shot            119            150        79.3                  25              50.0
  15-Shot            121            150        80.7  